# Experiment 2: Fine-tuned SAM-Med3D + TabPFN/LoCalPFN

This notebook fine-tunes SAM-Med3D on each dataset, then tests **TabPFN** and **LoCalPFN** with the fine-tuned features.

**Workflow:**
1. Prepare dataset in SAM-Med3D format
2. Fine-tune SAM-Med3D encoder (1-4 hours per dataset)
3. Extract features from fine-tuned encoder
4. Run TabPFN with fine-tuned features
5. Run LoCalPFN with fine-tuned features
6. Compare with pre-trained baseline

**Goal:** Measure performance improvement of TabPFN/LoCalPFN when using domain-adapted features.

In [1]:
# Environment setup
import os, sys, site, torch
os.environ['PYTHONNOUSERSITE'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # Fix OpenMP library conflict
usr = site.getusersitepackages(); sys.path = [p for p in sys.path if p != usr]

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
torch.set_num_threads(1)

from pathlib import Path
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            print('Added repo root to sys.path:', base)
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")

repo_root = _add_repo_root_to_sys_path()

Added repo root to sys.path: C:\Users\cahel\Desktop\Med3Tab-PFN


In [2]:
# Imports
from med3pipe.data.prepare import Sam3DPaths, find_default_sam3d_root, prepare_for_sam3d, split_validation
from med3pipe.training import finetune_sam3d
from med3pipe.pipelines import run_multi_tabpfn, run_multi_localpfn
import yaml
import pandas as pd
import time
import subprocess

config_path = repo_root / 'configs' / 'datasets.yaml'
outputs_base = repo_root / 'notebooks' / 'finetuned_pfn_results'
outputs_base.mkdir(exist_ok=True, parents=True)

# Select datasets
dataset_filter = ['gist', 'lipo']  # Or None for all

print('Config path:', config_path)
print('Outputs base:', outputs_base)

with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f) or {}
datasets_to_run = dataset_filter or list(cfg['datasets'].keys())
print('Datasets to run:', datasets_to_run)

# SAM-Med3D setup
sam3d_root = find_default_sam3d_root()
pretrained_checkpoint = sam3d_root / 'ckpt' / 'sam_med3d_turbo.pth'
if not pretrained_checkpoint.exists():
    pretrained_checkpoint = sam3d_root / 'ckpt' / 'SAM-Med3D-turbo.pth'
if not pretrained_checkpoint.exists():
    print("⚠️  WARNING: No pre-trained checkpoint found!")
    pretrained_checkpoint = None
else:
    print(f"✅ Pre-trained checkpoint: {pretrained_checkpoint}")

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config path: C:\Users\cahel\Desktop\Med3Tab-PFN\configs\datasets.yaml
Outputs base: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\finetuned_pfn_results
Datasets to run: ['gist', 'lipo']
✅ Pre-trained checkpoint: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth


## Configuration

In [ ]:
# Fine-tuning hyperparameters
FINETUNE_CONFIG = {
    'num_epochs': 3,           # Adjust: 2-5 for test, 20-50 for real
    'batch_size': 2,            # Adjust based on GPU memory
    'lr': 8e-5,
    'weight_decay': 0.1,
    'gpu_ids': [0],
    'multi_gpu': False,
    'device': 'cpu',           # 'cuda' or 'cpu' - SET TO 'cpu' IF NO GPU
}

# Check CUDA availability
if FINETUNE_CONFIG['device'] == 'cuda' and not torch.cuda.is_available():
    print("⚠️  CUDA not available, switching to CPU")
    FINETUNE_CONFIG['device'] = 'cpu'

print('Fine-tuning config:', FINETUNE_CONFIG)
print(f"Using device: {FINETUNE_CONFIG['device']}")

Fine-tuning config: {'num_epochs': 2, 'batch_size': 2, 'lr': 8e-05, 'weight_decay': 0.1, 'gpu_ids': [0], 'multi_gpu': False, 'device': 'cpu'}
Using device: cpu


In [4]:
def _resolve_dataset_root(dataset_root, category: str, project_root: Path) -> Path:
    if dataset_root is not None:
        p = Path(dataset_root)
        if p.is_absolute() and p.exists():
            return p
        cand = (project_root / p).resolve()
        if cand.exists():
            return cand
    c1 = (project_root / category).resolve()
    if c1.exists():
        return c1
    c2 = (project_root / 'data' / category).resolve()
    if c2.exists():
        return c2
    raise FileNotFoundError(f'Could not resolve dataset_root for {category!r}')

project_root = sam3d_root.parent.parent.resolve()
print('Project root:', project_root)

Project root: C:\Users\cahel\Desktop\Med3Tab-PFN


In [5]:
# Convert GPU checkpoint to CPU-compatible format
import torch
from pathlib import Path

gpu_ckpt = pretrained_checkpoint
cpu_ckpt = sam3d_root / 'ckpt' / 'sam_med3d_turbo_cpu.pth'

if gpu_ckpt and gpu_ckpt.exists() and not cpu_ckpt.exists():
    print(f"Converting GPU checkpoint to CPU format...")
    print(f"  Loading: {gpu_ckpt}")
    
    # Load with CPU map_location
    checkpoint = torch.load(str(gpu_ckpt), map_location='cpu', weights_only=False)
    
    # Save CPU version
    torch.save(checkpoint, str(cpu_ckpt))
    print(f"  ✅ Saved CPU checkpoint: {cpu_ckpt}")
    
    # Use CPU checkpoint for fine-tuning
    pretrained_checkpoint = cpu_ckpt
elif cpu_ckpt.exists():
    print(f"✅ CPU checkpoint already exists: {cpu_ckpt}")
    pretrained_checkpoint = cpu_ckpt
else:
    print("⚠️  No checkpoint to convert")

✅ CPU checkpoint already exists: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo_cpu.pth


In [6]:
# ============================================================================
# IMPORTANT: Convert GPU checkpoint to CPU format
# ============================================================================
import torch

gpu_ckpt = pretrained_checkpoint
cpu_ckpt = sam3d_root / 'ckpt' / 'sam_med3d_turbo_cpu.pth'

if gpu_ckpt and gpu_ckpt.exists() and not cpu_ckpt.exists():
    print(f"Converting GPU checkpoint to CPU format...")
    print(f"  Loading: {gpu_ckpt}")
    
    # Load with CPU map_location
    checkpoint = torch.load(str(gpu_ckpt), map_location='cpu', weights_only=False)
    
    # Save CPU version
    torch.save(checkpoint, str(cpu_ckpt))
    print(f"  ✅ Saved CPU checkpoint: {cpu_ckpt}")
    
    # Update the checkpoint path
    pretrained_checkpoint = cpu_ckpt
    print(f"  ✅ Updated pretrained_checkpoint to: {pretrained_checkpoint}")
elif cpu_ckpt.exists():
    print(f"✅ CPU checkpoint already exists: {cpu_ckpt}")
    pretrained_checkpoint = cpu_ckpt
else:
    print("⚠️  No checkpoint found")

print(f"\nFinal checkpoint to use: {pretrained_checkpoint}")

✅ CPU checkpoint already exists: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo_cpu.pth

Final checkpoint to use: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo_cpu.pth


In [7]:
# Use CPU checkpoint
cpu_ckpt = sam3d_root / 'ckpt' / 'sam_med3d_turbo_cpu.pth'
if cpu_ckpt.exists():
    pretrained_checkpoint = cpu_ckpt
    print(f"✅ Using CPU checkpoint: {pretrained_checkpoint}")
else:
    print(f"❌ CPU checkpoint not found at {cpu_ckpt}")

✅ Using CPU checkpoint: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo_cpu.pth


## Step 1: Fine-tune SAM-Med3D on Each Dataset

In [8]:
from typing import Dict, Any, List

finetuned_checkpoints: Dict[str, Path] = {}  # Store fine-tuned checkpoints
finetune_times: Dict[str, float] = {}

for ds_key in datasets_to_run:
    print(f"\n{'='*80}")
    print(f"Fine-tuning SAM-Med3D on: {ds_key}")
    print(f"{'='*80}\n")
    
    try:
        # Get dataset config
        ds_cfg = cfg['datasets'][ds_key] or {}
        category = ds_cfg.get('category', ds_key)
        ct_name = ds_cfg.get('ct_name', f'ct_{ds_key.upper()}')
        ds_root = _resolve_dataset_root(ds_cfg.get('dataset_root'), category=category, project_root=project_root)

        paths = Sam3DPaths(sam3d_root=sam3d_root, category=category, ct_name=ct_name)
        paths.ensure()
        
        # Prepare data if needed
        print(f"[1/2] Checking data...")
        if not (paths.images_tr).exists() or len(list(paths.images_tr.glob('*.nii.gz'))) == 0:
            print(f"   Preparing SAM-Med3D format...")
            n_prepared, _ = prepare_for_sam3d(
                dataset_root=ds_root,
                sam3d_root=sam3d_root,
                category=category,
                ct_name=ct_name,
            )
            # Create validation split
            split_validation(paths, split_ratio=0.8, seed=2025, copy=True)
        else:
            n_images = len(list(paths.images_tr.glob('*.nii.gz')))
            print(f"   ✓ Data already prepared: {n_images} images at {paths.train_root}")
        
        # Fine-tune
        print(f"\n[2/2] Fine-tuning SAM-Med3D...")
        print(f"   Device: {FINETUNE_CONFIG['device']}")
        print(f"   This will take 1-4 hours depending on dataset size, device, and epochs.")
        work_dir = outputs_base / f"{ds_key}_finetune_workdir"
        work_dir.mkdir(exist_ok=True, parents=True)
        
        # Create log file
        log_file = work_dir / f"{ds_key}_training.log"
        
        ft_start = time.time()
        


        # Force use of CPU checkpoint
        if FINETUNE_CONFIG['device'] == 'cpu':
            cpu_ckpt = sam3d_root / 'ckpt' / 'sam_med3d_turbo_cpu.pth'
            if cpu_ckpt.exists():
                pretrained_checkpoint_to_use = cpu_ckpt
            else:
                pretrained_checkpoint_to_use = pretrained_checkpoint
        else:
            pretrained_checkpoint_to_use = pretrained_checkpoint
        
        ft_start = time.time()
        proc = finetune_sam3d(
            paths=paths,
            sam3d_root=sam3d_root,
            checkpoint=pretrained_checkpoint_to_use,  # <-- Use this instead
            work_dir=work_dir,
            **FINETUNE_CONFIG
        )


        # Call finetune - it returns a Popen object
        proc = finetune_sam3d(
            paths=paths,
            sam3d_root=sam3d_root,
            checkpoint=pretrained_checkpoint,
            work_dir=work_dir,
            **FINETUNE_CONFIG
        )
        
        print(f"   Training process started (PID: {proc.pid})") 
        print(f"   Logs will be in: {work_dir}")
        print(f"   Monitoring output...")
        print("   " + "="*76)
        
        # Read output in real-time
        for line in proc.stdout:
            print(f"   {line.rstrip()}")
        
        # Wait for process to complete
        returncode = proc.wait()
        ft_duration = time.time() - ft_start
        
        print("   " + "="*76)
        print(f"   ✅ Fine-tuning completed in {ft_duration/60:.1f} minutes")
        
        # Find best checkpoint
        task_name = f"{category}_{ct_name}_ft"
        task_dir = work_dir / task_name
        best_ckpt = task_dir / "sam_model_dice_best.pth"
        if not best_ckpt.exists():
            best_ckpt = task_dir / "sam_model_latest.pth"
        if not best_ckpt.exists():
            raise FileNotFoundError(f"No checkpoint found in {task_dir}")
        
        finetuned_checkpoints[ds_key] = best_ckpt
        finetune_times[ds_key] = ft_duration / 60
        print(f"   Checkpoint saved: {best_ckpt}")
        
    except Exception as e:
        print(f"\n❌ Error fine-tuning {ds_key}: {e}")
        import traceback
        traceback.print_exc()
        finetuned_checkpoints[ds_key] = None
        finetune_times[ds_key] = None

print(f"\n{'='*80}")
print(f"Fine-tuning complete for all datasets")
print(f"{'='*80}")
print("\nFine-tuned checkpoints:")
for ds, ckpt in finetuned_checkpoints.items():
    print(f"  {ds}: {ckpt}")


Fine-tuning SAM-Med3D on: gist

[1/2] Checking data...
   ✓ Data already prepared: 246 images at C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST

[2/2] Fine-tuning SAM-Med3D...
   Device: cpu
   This will take 1-4 hours depending on dataset size, device, and epochs.
   Training process started (PID: 31460)
   Logs will be in: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\finetuned_pfn_results\gist_finetune_workdir
   Monitoring output...
   c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torch\utils\data\dataloader.py:624: UserWarning: This DataLoader will create 24 worker processes in total. Our suggested max number of worker in current system is 20 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
     warnings.warn(
   

Traceback (most recent call last):
  File "C:\Users\cahel\AppData\Local\Temp\ipykernel_23220\2401177300.py", line 103, in <module>
    raise FileNotFoundError(f"No checkpoint found in {task_dir}")
FileNotFoundError: No checkpoint found in C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\finetuned_pfn_results\gist_finetune_workdir\gist_ct_GIST_ft


   c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torch\utils\data\dataloader.py:624: UserWarning: This DataLoader will create 24 worker processes in total. Our suggested max number of worker in current system is 20 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
     warnings.warn(
   Loaded checkpoint from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo_cpu.pth (epoch 0)
   c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
     warnings.warn(
   Epoch: 0/1
   
     0%|          | 0/123 [00:00<?, ?it/s]c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: 

Traceback (most recent call last):
  File "C:\Users\cahel\AppData\Local\Temp\ipykernel_23220\2401177300.py", line 103, in <module>
    raise FileNotFoundError(f"No checkpoint found in {task_dir}")
FileNotFoundError: No checkpoint found in C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks\finetuned_pfn_results\lipo_finetune_workdir\lipo_ct_LIPO_ft


## Step 2: Run TabPFN with Fine-tuned Features

In [9]:
print("\n" + "="*80)
print("Running TabPFN with FINE-TUNED SAM-Med3D features")
print("="*80 + "\n")

# We'll run TabPFN once for each fine-tuned checkpoint
tabpfn_results = []

for ds_key in datasets_to_run:
    ft_ckpt = finetuned_checkpoints.get(ds_key)
    if ft_ckpt is None or not Path(ft_ckpt).exists():
        print(f"⚠️  Skipping {ds_key}: no fine-tuned checkpoint")
        continue
    
    print(f"\n--- TabPFN on {ds_key} with fine-tuned features ---")
    
    try:
        res = run_multi_tabpfn(
            config_path=config_path,
            dataset_names=[ds_key],
            outputs_base_dir=outputs_base / f"tabpfn_finetuned",
            checkpoint=ft_ckpt,  # Use fine-tuned checkpoint
            skip_existing_embeddings=False,  # Re-extract with fine-tuned model
        )
        tabpfn_results.append(res['summary_df'])
    except Exception as e:
        print(f"❌ Error running TabPFN on {ds_key}: {e}")
        import traceback
        traceback.print_exc()

df_tabpfn_finetuned = pd.concat(tabpfn_results, ignore_index=True) if tabpfn_results else pd.DataFrame()
print("\n✅ TabPFN with fine-tuned features complete")
df_tabpfn_finetuned


Running TabPFN with FINE-TUNED SAM-Med3D features

⚠️  Skipping gist: no fine-tuned checkpoint
⚠️  Skipping lipo: no fine-tuned checkpoint

✅ TabPFN with fine-tuned features complete


""


## Step 3: Run LoCalPFN with Fine-tuned Features

In [10]:
print("\n" + "="*80)
print("Running LoCalPFN with FINE-TUNED SAM-Med3D features")
print("="*80 + "\n")

localpfn_results = []

for ds_key in datasets_to_run:
    ft_ckpt = finetuned_checkpoints.get(ds_key)
    if ft_ckpt is None or not Path(ft_ckpt).exists():
        print(f"⚠️  Skipping {ds_key}: no fine-tuned checkpoint")
        continue
    
    print(f"\n--- LoCalPFN on {ds_key} with fine-tuned features ---")
    
    try:
        res = run_multi_localpfn(
            config_path=config_path,
            dataset_names=[ds_key],
            outputs_base_dir=outputs_base / f"localpfn_finetuned",
            checkpoint=ft_ckpt,  # Use fine-tuned checkpoint
            skip_existing_embeddings=False,  # Re-extract with fine-tuned model
            local_k=8,
            local_fit_adapter=True,
            local_adapter_epochs=8,
        )
        localpfn_results.append(res['summary_df'])
    except Exception as e:
        print(f"❌ Error running LoCalPFN on {ds_key}: {e}")
        import traceback
        traceback.print_exc()

df_localpfn_finetuned = pd.concat(localpfn_results, ignore_index=True) if localpfn_results else pd.DataFrame()
print("\n✅ LoCalPFN with fine-tuned features complete")
df_localpfn_finetuned


Running LoCalPFN with FINE-TUNED SAM-Med3D features

⚠️  Skipping gist: no fine-tuned checkpoint
⚠️  Skipping lipo: no fine-tuned checkpoint

✅ LoCalPFN with fine-tuned features complete


""


## Results Summary

In [11]:
# Combine results
df_combined = pd.concat([
    df_tabpfn_finetuned,
    df_localpfn_finetuned
], ignore_index=True)

# Add method suffix to distinguish from pre-trained
df_combined['method'] = df_combined['method'] + '_finetuned'

# Add fine-tuning time
df_combined['finetune_time_min'] = df_combined['dataset'].map(finetune_times)

print("\n📊 Results with Fine-tuned SAM-Med3D\n")
df_combined

KeyError: 'method'

In [ ]:
# Save results
out_csv = outputs_base / 'finetuned_pfn_results.csv'
df_combined.to_csv(out_csv, index=False)
print(f'✅ Saved results to: {out_csv}')